# 🛠️ Notebook 02: Verificación del Preprocesamiento
**Proyecto:** Análisis de Sentimiento Bayesiano con Calibración de Confianza

Este notebook tiene como objetivo validar la integridad de las representaciones vectoriales generadas. Comprobaremos:
1. **Dimensiones:** Que los vectores de BERT y TF-IDF tengan el tamaño esperado.
2. **Integridad:** Ausencia de valores nulos o infinitos que rompan la Inferencia Variacional.
3. **Loader:** Verificación de que la clase `BNNDataset` puede servir datos correctamente a la BNN.

In [1]:
import os
import numpy as np
import scipy.sparse as sp
import torch

# Ajuste de ruta para notebooks
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Carga de datos procesados
bert_embs = np.load("data/processed/bert_embeddings.npy")
tfidf_mat = sp.load_npz("data/processed/tfidf_matrix.npz")
labels = np.load("data/processed/labels.npy")

print(f"✅ BERT Embeddings cargados: {bert_embs.shape} (Esperado: N x 768)")
print(f"✅ TF-IDF Matrix cargada: {tfidf_mat.shape} (Esperado: N x 10000)")
print(f"✅ Etiquetas cargadas: {labels.shape}")

✅ BERT Embeddings cargados: (10500, 768) (Esperado: N x 768)
✅ TF-IDF Matrix cargada: (10500, 10000) (Esperado: N x 10000)
✅ Etiquetas cargadas: (10500,)


In [2]:
# Comprobamos que no haya NaNs (muy importante para Pyro/Inferencia Variacional)
assert not np.isnan(bert_embs).any(), "🚨 Error: Se han detectado NaNs en los embeddings de BERT."
assert not np.isinf(bert_embs).any(), "🚨 Error: Se han detectado valores infinitos en los embeddings."

# Verificar que las etiquetas ID son binarias (0 o 1)
id_labels = labels[labels != -1]
unique_labels = np.unique(id_labels)
print(f"Valores únicos en etiquetas ID: {unique_labels} (Debe ser [0 1])")

Valores únicos en etiquetas ID: [0 1] (Debe ser [0 1])


In [3]:
import sys
import os

# Aseguramos que la raíz del proyecto esté en el path para que los workers encuentren 'src'
root_path = os.path.abspath(os.path.join(os.getcwd()))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data.loader import make_dataloaders

try:
    # IMPORTANTE: num_workers=0 en el loader.py evita errores de spawn en macOS
    train_loader, val_loader, _ = make_dataloaders(batch_size=16)
    
    x_sample, y_sample = next(iter(train_loader))
    
    print("--- Test de DataLoader Exitoso ---")
    print(f"Dimensiones del batch X: {x_sample.shape}") # [16, 768]
    print(f"Dimensiones del batch y: {y_sample.shape}") # [16]
    print("✅ Loader funcionando correctamente.")
    
except Exception as e:
    print(f"❌ Error en el loader: {e}")

--- Test de DataLoader Exitoso ---
Dimensiones del batch X: torch.Size([16, 768])
Dimensiones del batch y: torch.Size([16])
✅ Loader funcionando correctamente.
